In [ ]:
%sh
pwd
echo "----- contenido del CWD -----"
ls -la
echo "----- buscar los .py en todo el checkout -----"
find /Workspace/Repos/.internal -name "*.py" 2>/dev/null | head -50

In [ ]:
import os, sys, json
from importlib.util import find_spec

rep = {}
rep["cwd"] = os.getcwd()
try:
    rep["listdir"] = sorted(os.listdir(os.getcwd()))
except Exception as e:
    rep["listdir"] = "ERR " + str(e)
# WHO is executing? group down-scoping check
try:
    rep["current_user"] = spark.sql("select current_user() as u").collect()[0]["u"]
except Exception as e:
    rep["current_user"] = "ERR " + str(e)
try:
    rep["is_member_demo"] = spark.sql("select is_member('demo_git_folder') as m").collect()[0]["m"]
except Exception as e:
    rep["is_member_demo"] = "ERR " + str(e)
# can we WRITE to the checkout dir? (the doc's claim)
try:
    p = os.path.join(os.getcwd(), "_write_probe.txt")
    open(p, "w").write("probe")
    rep["checkout_writable"] = True
    os.remove(p)
except Exception as e:
    rep["checkout_writable"] = "NO: " + type(e).__name__ + ": " + str(e)[:200]
spec = find_spec("funciones")
rep["find_spec"] = str(spec)
try:
    from funciones import transformar_datos
    rep["import"] = "OK"; rep["result"] = transformar_datos(21)
except Exception as e:
    rep["import"] = "FAIL: " + type(e).__name__ + ": " + str(e)[:200]
print(json.dumps(rep, indent=2))
dbutils.notebook.exit(json.dumps(rep))